# ML Model Training Notebook
## Employee Performance Prediction using Random Forest

This notebook demonstrates the complete ML pipeline for the AI-Enhanced Employee Assessment Platform.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

print('All libraries imported successfully!')

In [ ]:
# Generate synthetic dataset
np.random.seed(42)
n_samples = 1000

data = []
samples_per_class = n_samples // 3

# High Performers
for _ in range(samples_per_class):
    score = np.random.uniform(75, 100)
    accuracy = np.random.uniform(80, 100)
    time_taken = np.random.uniform(300, 900)
    attempts = np.random.choice([1, 2], p=[0.7, 0.3])
    previous_performance = np.random.uniform(70, 100)
    data.append([score, accuracy, time_taken, attempts, previous_performance, 'High Performer'])

# Average Performers
for _ in range(samples_per_class):
    score = np.random.uniform(50, 80)
    accuracy = np.random.uniform(55, 85)
    time_taken = np.random.uniform(400, 1200)
    attempts = np.random.choice([1, 2, 3], p=[0.4, 0.4, 0.2])
    previous_performance = np.random.uniform(50, 80)
    data.append([score, accuracy, time_taken, attempts, previous_performance, 'Average Performer'])

# Needs Improvement
for _ in range(n_samples - 2 * samples_per_class):
    score = np.random.uniform(20, 55)
    accuracy = np.random.uniform(25, 60)
    time_taken = np.random.uniform(600, 1500)
    attempts = np.random.choice([1, 2, 3, 4], p=[0.2, 0.3, 0.3, 0.2])
    previous_performance = np.random.uniform(30, 60)
    data.append([score, accuracy, time_taken, attempts, previous_performance, 'Needs Improvement'])

df = pd.DataFrame(data, columns=['score', 'accuracy', 'time_taken', 'attempts', 'previous_performance', 'performance_category'])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['performance_category'].value_counts())
df.head()

In [ ]:
# Exploratory Data Analysis
print('Dataset Info:')
print(df.info())

print('\nMissing Values:')
print(df.isnull().sum())

print('\nStatistical Summary:')
print(df.describe())

# Visualize distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features = ['score', 'accuracy', 'time_taken', 'attempts', 'previous_performance']
colors = {'High Performer': 'green', 'Average Performer': 'orange', 'Needs Improvement': 'red'}

for idx, feature in enumerate(features):
    ax = axes[idx // 3][idx % 3]
    for category in df['performance_category'].unique():
        subset = df[df['performance_category'] == category]
        ax.hist(subset[feature], alpha=0.5, label=category, bins=20, color=colors[category])
    ax.set_title(f'{feature} Distribution')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing
label_encoder = LabelEncoder()
df['encoded_category'] = label_encoder.fit_transform(df['performance_category'])

X = df[['score', 'accuracy', 'time_taken', 'attempts', 'previous_performance']].values
y = df['encoded_category'].values

print('Features shape:', X.shape)
print('Target shape:', y.shape)
print('Classes:', label_encoder.classes_)

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples: {X_test.shape[0]}')

In [ ]:
# Train Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
print('Model training completed!')

In [ ]:
# Model Evaluation
y_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# Feature Importance
feature_names = ['score', 'accuracy', 'time_taken', 'attempts', 'previous_performance']
importances = rf_model.feature_importances_

plt.figure(figsize=(10, 6))
sorted_idx = np.argsort(importances)
plt.barh(range(len(sorted_idx)), importances[sorted_idx])
plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.show()

print('\nFeature Importance:')
for name, imp in sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True):
    print(f'{name}: {imp:.4f}')

In [ ]:
# Save Model
model_data = {
    'model': rf_model,
    'label_encoder': label_encoder,
    'feature_names': feature_names,
    'class_names': label_encoder.classes_.tolist()
}

joblib.dump(model_data, 'employee_model.pkl')
print('Model saved as employee_model.pkl')

# Save dataset
df.to_csv('dataset/employee_performance.csv', index=False)
print('Dataset saved as dataset/employee_performance.csv')

In [ ]:
# Test Prediction
loaded_data = joblib.load('employee_model.pkl')
loaded_model = loaded_data['model']
loaded_encoder = loaded_data['label_encoder']

# Sample prediction
sample_features = np.array([[85, 90.0, 400, 1, 80.0]])
prediction = loaded_model.predict(sample_features)
pred_class = loaded_encoder.inverse_transform(prediction)[0]
probs = loaded_model.predict_proba(sample_features)[0]
confidence = max(probs)

print(f'Sample Features: score=85, accuracy=90%, time=400s, attempts=1, prev=80%')
print(f'Predicted Class: {pred_class}')
print(f'Confidence: {confidence:.2%}')
print(f'Class Probabilities: {dict(zip(loaded_encoder.classes_, probs))}')